In [14]:
import numpy as np
import random

In [16]:
# --- 1. CONFIGURATION ---
ALPHA = 0.1
GAMMA = 0.9
EPSILON = 0.2
EPOCHS = 5000

MOVE_AMOUNT = 20 
TOTAL_CAPACITY = 1000

# tiers
TIERS = ['A', 'B', 'C', 'D']

# Actions: 0: Stay, 1: D->A, 2: C->A, ...
ACTIONS = {
    0: "Stay",
    1: ("D", "A"), 2: ("C", "A"), 3: ("B", "A"), # Moves to A
    4: ("D", "B"), 5: ("C", "B"), 6: ("A", "B"), # Moves to B
    7: ("D", "C"), 8: ("B", "C"), 9: ("A", "C"), # Moves to C
    10: ("C", "D"), 11: ("B", "D"), 12: ("A", "D") # Moves to D
}

# 3 (idle, balanced, pressure)
STATUS = 3

# 3^4 States (81 total)
""" 
Tier A: 3 possible status (Idle, Balanced, Pressure)
Tier B: 3 possible status
Tier C: 3 possible status
Tier D: 3 possible status
"""
STATES = [(a, b, c, d) for a in range(STATUS) for b in range(STATUS) for c in range(STATUS) for d in range(STATUS)]
q_table = np.zeros((len(STATES), len(ACTIONS)))

print("q_table")
print(q_table)

q_table
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [18]:
def get_tier_status(usage, limit):
    ratio = usage / limit
    if ratio > 0.8: return 2   # Under Pressure
    if ratio < 0.4: return 0   # Idle 
    return 1                   # Balanced

def get_state_index(usage_dict, limit_dict, verbose=False):
    # 1. Simplify the 4 tiers into statuses (0, 1, or 2)
    statuses = []
    for t in TIERS:
        status = get_tier_status(usage_dict[t], limit_dict[t])
        statuses.append(status)
    
    # 2. Create the tuple (The 'Snapshot')
    st = tuple(statuses)
    
    # 3. Find the index in our 81-state map
    idx = STATES.index(st)
    
    if verbose:
        print(f"   --- Decision Logic ---")
        print(f"   Current Lim:   {limit_dict}")
        print(f"   Current Usage: {usage_dict}")
        print(f"   State Tuple: {st} (Index: {idx})")
        # Translation of the tuple for humans
        status_map = {0: "IDLE", 1: "OK", 2: "PRESSURE"}
        readable = [f"{TIERS[i]}:{status_map[st[i]]}" for i in range(4)]
        print(f"   Readable:    {' | '.join(readable)}")
        
    return idx

def apply_action(action_idx, current_limits):
    new_limits = current_limits.copy()
    if action_idx == 0:
        return new_limits
    
    source, target = ACTIONS[action_idx]
    
    # Check if source has enough to give (Minimum floor of 40 RPS per tier)
    if new_limits[source] >= (25 + MOVE_AMOUNT):
        new_limits[source] -= MOVE_AMOUNT
        new_limits[target] += MOVE_AMOUNT
        
    # Validation: Ensure we haven't broken the 1000 RPS total
    assert sum(new_limits.values()) == TOTAL_CAPACITY, "Capacity mismatch!"
    return new_limits

def calculate_reward(usage, limits):
    reward = 0
    for t in TIERS:
        # Penalty for Pressure (>70%)
        if usage[t] / limits[t] > 0.8:
            reward -= 20
        # Penalty for Waste (<30%)
        if usage[t] / limits[t] < 0.4:
            reward -= 10
        # Critical Penalty for Starvation (Usage > Limit)
        if usage[t] > limits[t]:
            reward -= 100
    return reward

In [19]:
scenarios = [     
    {'A': 480, 'B': 200, 'C': 50, 'D': 40},  # A needs help from C or D
    {'A': 100, 'B': 380, 'C': 50, 'D': 40}, # B needs help from A or D
    {'A': 100, 'B': 200, 'C': 280, 'D': 40}, # B needs help from A or D    
    {'A': 250, 'B': 250, 'C': 250, 'D': 250}, # Already balanced
    #{'A': 480, 'B': 300, 'C': 80, 'D': 50},  # A needs help from C or D
    #{'A': 100, 'B': 380, 'C': 250, 'D': 40}, # B needs help from A or D
    #{'A': 250, 'B': 250, 'C': 250, 'D': 250}, # Already balanced
]

In [27]:
EPOCHS = 1000
verbose=False

for _ in range(EPOCHS):
    # choose random from scenario
    choosen_scenario = random.choice(scenarios)

    # Start limits
    limit = {'A': 400, 'B': 300, 'C': 200, 'D': 100} 

    if verbose:
        print("choosen_scenario:", choosen_scenario)
     
    for _step in range(20): # Allow 20 moves per epoch to find balance

        if verbose:
            print(".._step:",_step)

        # Pick the state 
        if verbose:
            print("..-- Pick the state --")
        """ 
        Raw Usage:  {'A': 100, 'B': 380, 'C': 250, 'D': 40}
        Current Lim: {'A': 400, 'B': 300, 'C': 200, 'D': 100}
        State Tuple: (0, 2, 2, 1) (Index: 25)
        Readable:    A:IDLE | B:PRESSURE | C:PRESSURE | D:OK
        """        
        state_idx = get_state_index(choosen_scenario, limit, verbose)
            
        if verbose:                            
            print("..state_idx:",state_idx)

        # Pick the highest value from q_table[state_idx]
        # action: index 5 (-144.13828548) is the highest = [-169.64210954 -155.96432577 -146.10314095 -157.99391262 -165.7516957 -144.13828548 -168.96336113 -160.23067013 -146.2996265 ]
        if random.uniform(0, 1) < EPSILON:
            action = random.randint(0, len(ACTIONS)-1)
            if verbose:
                print(".. *EXPLORATION*")
        else:
            action = np.argmax(q_table[state_idx])

        if verbose:
            print("..action:", action, "=" ,q_table[state_idx])

        # Apply the action (balance rps from tiers)
        new_limit = apply_action(action, limit)
        
        if verbose:
            print("..new_limit:", new_limit)

        # Calc the reward according the new limit
        reward = calculate_reward(choosen_scenario, new_limit)

        if verbose:
            print("..reward:", reward)

        # Pick the next state

        if verbose:
            print(".. *Pick the next state*")
        
        next_s_idx = get_state_index(choosen_scenario, new_limit, verbose)
        
        if verbose:
            print("..next_s_idx:", next_s_idx)
        
        # Q-Update
        q_table[state_idx, action] += ALPHA * (reward + GAMMA * np.max(q_table[next_s_idx]) - q_table[state_idx, action])

        if verbose:        
            print(f"..q_table[state_idx {state_idx}, action {action}] :", q_table[state_idx, action], q_table[state_idx] )
        
        limit = new_limit


In [28]:
print("q_table")
#print(q_table[40])
print(q_table)

q_table
[[    0.             0.             0.         ...     0.
      0.             0.        ]
 [    0.             0.             0.         ...     0.
      0.             0.        ]
 [    0.             0.             0.         ...     0.
      0.             0.        ]
 ...
 [ -196.75566773  -196.33563072  -202.89541383 ...  -220.82548652
   -199.11001662  -197.0728137 ]
 [    0.             0.             0.         ...     0.
      0.             0.        ]
 [-1277.57946475 -1292.4388463  -1310.45543016 ... -1267.85530357
  -1285.22922996 -1260.68681983]]


In [29]:
test_limits = {'A': 400, 'B': 300, 'C': 200, 'D': 100} 
test_usage = {'A': 100, 'B': 250, 'C': 250, 'D': 130}

test_limits = {'A': 500, 'B': 300, 'C': 150, 'D': 50}
test_usage = {'A': 490, 'B': 250, 'C': 50, 'D': 40} # A needs help from C

print(f"Initial Limit: {test_limits}")
print("Agent is rebalancing...")

for i in range(30):
    idx = get_state_index(test_usage, test_limits)
    best_action = np.argmax(q_table[idx])
    test_limits = apply_action(best_action, test_limits)
    action_name = ACTIONS[best_action]
    print(f"Step {i+1}: Action {action_name} -> New Limits: {test_limits}")

print(f"\nFinal Total Capacity: {sum(test_limits.values())} RPS")

Initial Limit: {'A': 500, 'B': 300, 'C': 150, 'D': 50}
Agent is rebalancing...
Step 1: Action ('C', 'A') -> New Limits: {'A': 520, 'B': 300, 'C': 130, 'D': 50}
Step 2: Action ('C', 'A') -> New Limits: {'A': 540, 'B': 300, 'C': 110, 'D': 50}
Step 3: Action ('A', 'B') -> New Limits: {'A': 520, 'B': 320, 'C': 110, 'D': 50}
Step 4: Action ('B', 'A') -> New Limits: {'A': 540, 'B': 300, 'C': 110, 'D': 50}
Step 5: Action ('A', 'B') -> New Limits: {'A': 520, 'B': 320, 'C': 110, 'D': 50}
Step 6: Action ('B', 'A') -> New Limits: {'A': 540, 'B': 300, 'C': 110, 'D': 50}
Step 7: Action ('A', 'B') -> New Limits: {'A': 520, 'B': 320, 'C': 110, 'D': 50}
Step 8: Action ('B', 'A') -> New Limits: {'A': 540, 'B': 300, 'C': 110, 'D': 50}
Step 9: Action ('A', 'B') -> New Limits: {'A': 520, 'B': 320, 'C': 110, 'D': 50}
Step 10: Action ('B', 'A') -> New Limits: {'A': 540, 'B': 300, 'C': 110, 'D': 50}
Step 11: Action ('A', 'B') -> New Limits: {'A': 520, 'B': 320, 'C': 110, 'D': 50}
Step 12: Action ('B', 'A') -